### Export coefficeints to file

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from chimorse.config import (
    FigureContext, get_colors, PLOT_PARAMS, MOLECULES
)
from chimorse.dataio import load_data, export_chimorse_single_json
from chimorse.fitting import _parse_prune_arg, extract_reduced_coeffs, fit_alpha_values
from chimorse.fourier import create_matrix_lsqt_2d
from chimorse.analysis import extract_energy_minimums, get_symm_chi

plt.rcParams.update(PLOT_PARAMS)

In [ ]:
molecule_name = 'PA'
interaction = 'OA'
zero_zeta = True

alpha_fit = True

harmonic_ceils = {'EP': (8, 1), 'EA': (8, 1), 'OP': (20, 1), 'OA': (20, 1)}

molecule = MOLECULES[molecule_name]

df_org = load_data(molecule, interaction, zero_zeta)
colors = get_colors()

ctx_org = FigureContext(base="Figures", molecule=molecule.name, data_type="org", interaction=interaction)
ctx_model = FigureContext(base="Figures", molecule=molecule.name, data_type="model", interaction=interaction)
ctx_compare = FigureContext(base="Figures", molecule=molecule.name, data_type="compare", interaction=interaction)

In [ ]:
alpha_fit = False
# Optional: set True to fit alpha(chi, psi) via Fourier expansion instead of a fixed value.

# Magnitude-based pruning of the least-squares coefficients. Exactly one of the two
# options below must be set (both to None to skip pruning and export full coefficients):
prune_relative_th = 1e-3
#   keep coefficients with |coeff| > prune_relative_th * max|coeff|
prune_top_n = None
#   keep only the top-N largest-magnitude coefficients, e.g. {'D': 40, 're': 45, 'alpha': 80}

thresh = _parse_prune_arg(prune_relative_th)
top = _parse_prune_arg(prune_top_n)

for interaction in ['EP', 'EA', 'OP', 'OA']:
    df_org = load_data(molecule, interaction, zero_zeta)

    E_min_df = extract_energy_minimums(df_org, r_max=12)
    D, re = -E_min_df['e'], E_min_df['r']
    chi_rad, psi_rad = np.deg2rad(E_min_df['chi']), np.deg2rad(E_min_df['psi'])
    h_chi, h_psi = harmonic_ceils[interaction]
    symm_chi = get_symm_chi(interaction)

    A, labels = create_matrix_lsqt_2d(h_chi, h_psi, chi_rad, psi_rad, symm_chi, molecule.screw_step)

    D_coeff, *_ = np.linalg.lstsq(A, D, rcond=None)
    re_coeff, *_ = np.linalg.lstsq(A, re, rcond=None)
    alpha_coeff = None

    if alpha_fit:
        alpha_vals = fit_alpha_values(df_org, interaction)
        alpha_coeff, *_ = np.linalg.lstsq(A, alpha_vals, rcond=None)

    if any(t is not None for t in thresh.values()) or any(t is not None for t in top.values()):
        D_coeff, _ = extract_reduced_coeffs(A, D, threshold=thresh['D'], top_n=top['D'])
        re_coeff, _ = extract_reduced_coeffs(A, re, threshold=thresh['re'], top_n=top['re'])
        if alpha_fit:
            alpha_coeff, _ = extract_reduced_coeffs(A, alpha_vals, threshold=thresh['alpha'], top_n=top['alpha'])

    export_chimorse_single_json(
        filepath=f"models/{interaction}.json",
        h_chi=h_chi,
        h_psi=h_psi,
        symm_chi=symm_chi,
        screw_step=molecule.screw_step,
        D_coeff=D_coeff,
        re_coeff=re_coeff,
        fixed_alpha=1.1 if alpha_coeff is None else None,
        alpha_coeff=alpha_coeff,
        cutoff=12.0,
    )


In [ ]:
from chimorse.dataio import export_chimorse_combined_json

export_chimorse_combined_json(
    target_file="models/chimorse_all.json",
    source_directory="models",
)

---

### Calculate Force and export Potential/ Force to C++

In [ ]:
import sympy as sp
from chimorse.models import Morse_1D

r, dphi = sp.symbols('r deltaPhi', real=True)
D, re, a, h, alpha = sp.symbols('D re a h alpha', real=True)

morse_part = D * (sp.exp(-2 * a * (r - re)) - 2 * sp.exp(-a * (r - re)))
fourier_part = sp.cos(h * dphi + alpha) # example

U_term = morse_part * fourier_part

F_radial = -sp.diff(U_term, r)
F_angular = -sp.diff(U_term, dphi)

print("====== C++ Code for Potential Energy ======")
print(sp.ccode(U_term, assign_to="energy_term"))

print("\n====== C++ Code for Radial Force ======")
print(sp.ccode(F_radial, assign_to="radial_term"))

print("\n====== C++ Code for Angular Force ======")
print(sp.ccode(F_angular, assign_to="angular_term"))